In [1]:
#req
import re
import pandas as pd
from datetime import time

#functie om verder in het script breukvormen te harmoniseren
def clean_fraction(time_str):
    return re.sub(r'(\d+)\s+(\d+)\s*/\s*(\d+)', r'\1 \2/\3', time_str) 

#hoofdcomponent van het script
def normalize_time(time_str):
    if pd.isna(time_str) or str(time_str).strip() == '':
        return None
    
    original = str(time_str).lower().strip()
    time_str = clean_fraction(original)
    
    #detectie van dagdelen in tekstuele segmenten op basis van patroonherkenning
    time_of_day_patterns = {
        'morning': [
            r'\b(?:matin|mati|matine)\b',
            r'\b(?:morgend|morgens|smorgends|s\'morgends|smogends|s\'mogends|voormiddag|ochtend|hormiddag|\'s\s*morgen|s\s*morgen)\b',
            r'ochtend',
            r'\bmorgen\b'
        ],
        'noon': [
            r'\b(?:midi|mid-day|middag|midag|middags?)\b'
        ],
        'afternoon': [
            r'ap(?:r[eè]s|i[eè]s)[\s-]*midi\b',
            r'apr[i]?[e]?[s]?[\s-]*m[ui]t[i]?\b',  
            r'l[\'\s]ap[ri][eè]s[\s-]*m[iu][dr]i\b',
            r'de\s+l[i\']p+r[ei]+s[\s-]*midi\b',    
            r'apris[\s-]*midi',                     
            r'n\s*de\s*l[\'\s]ap[ri][eè]s[\s-]*midi',
            r'.*apr[ieè]s[\s-]*midi.*', 
            r'(?:na[\s-]*(?:mid|nid|tid)[\s-]*dag|namidd|naar?midd?ag)\b',
            r'\bnamidd?ags?\b',
            r'\'?\s*s\s*namidd?ags?\b',
            r'\'?\s+s\s+namidd?ags?\b',
            r'\bjour\b'
        ],
        'evening': [
            r'\b(?:soir|soire|soiree|soirée)\b',
            r'\b(?:avond|savonds|s\'avonds|avons)\b',
            r'[\'\s]s\s*avond',
            r's\s*[\'\`]?\s*avonds?',
            r'\bavond\b'
        ],
        'night': [
            r'\b(?:nuit|nuite)\b',
            r'\b(?:nacht|nachts|snachts|nachtdienst|nact)\b',
            r'[\'\s]s\s*nachts'
        ]
    }
    
    print(f"Original time string: {original}")
    
    parts = time_str.split(';') #afonderlijk herkende entiteiten worden afzonderlijk behandeld
    time_of_day = None
    
    #omwille van veelvoorkomende problemen met "après-midi" wordt hier voorrang aan gegeven
    for part in parts:
        print(f"Checking part: {part}")
        if re.search(r'apr[ieès]s[\s-]*midi', part.lower()):
            time_of_day = 'afternoon'
            print(f"Special case - found afternoon pattern 'apres-midi'")
            break

    #als er bovenstaande patronen niet in de tijdsnotering worden gevonden: verder zoeken via bovenstaande lijst met patronen        
    if not time_of_day:
        for part in parts:
            for period, patterns in time_of_day_patterns.items():
                for pattern in patterns:
                    if re.search(pattern, part):
                        time_of_day = period
                        print(f"Found time of day: {period} using pattern: {pattern}")
                        break
                if time_of_day:
                    break
            if time_of_day:                break

    print(f"Time of day detected: {time_of_day}")
    
    #stap 2: zoek numerieke segmenten
    time_part = None
    for part in parts:
        if re.search(r'\d', part):
            time_part = part.strip()
            break
    
    if time_part is None:
        time_part = parts[-1].strip()
    
    print(f"Time part: {time_part}")
    
    has_explicit_hours = False
    hours = 0
    minutes = 0
    
    #vroege foutdetectie: foutieve breukvormen zoals '13/4' in de ochtend
    problematic_pattern = re.search(r'^\s*(\d+)\s*\/\s*(\d+)', time_part)
    if problematic_pattern:
        try:
            potential_hours = int(problematic_pattern.group(1))
            if time_of_day == 'morning' and potential_hours >= 12:
                print(f"Detected invalid morning hours ({potential_hours}) from pattern. Using default morning time.")
                return '09:00'
        except ValueError:
            pass
  
    complex_pattern = re.search(r'^\s*(\d+)\s+(\d+)\s*\/\s*(\d+)', time_part)
    if complex_pattern:
        try:
            potential_hours = int(complex_pattern.group(1))
            if time_of_day == 'morning' and potential_hours >= 12:
                print(f"Detected invalid morning hours ({potential_hours}) from complex pattern. Using default morning time.")
                return '09:00'
        except ValueError:
            pass
    
    #herken gecombineerde breuken zoals "71/4"  
    joined_hour_fraction = re.search(r'^(\d)(\d)\s*\/\s*(\d+)', time_part)
    if joined_hour_fraction:
        hours = int(joined_hour_fraction.group(1))
        numerator = int(joined_hour_fraction.group(2))
        denominator = int(joined_hour_fraction.group(3))
        has_explicit_hours = True
        minutes = (numerator * 60) // denominator
        print(f"Joined hour fraction: {hours} hours and {minutes} minutes")
    
    #Franse voluit geschreven uren
    if not joined_hour_fraction:
        french_hour_words = {
            r'une\s+heure': 1, r'deux\s+heures': 2, r'trois\s+heures': 3, r'quatre\s+heures': 4,
            r'cinq\s+heures': 5, r'six\s+heures': 6, r'sept\s+heures': 7, r'huit\s+heures': 8,
            r'neuf\s+heures': 9, r'dix\s+heures': 10, r'onze\s+heures': 11, r'douze\s+heures': 12
        }
        
        for pattern, value in french_hour_words.items():
            if re.search(pattern, time_part):
                hours = value
                has_explicit_hours = True
                print(f"French hour words: {hours} hours")
                break
    
    #Nederlandstalige "een uur"
    if not has_explicit_hours and not joined_hour_fraction and re.search(r'een\s+u+[re]+', time_part):
        hours = 1
        has_explicit_hours = True
        print("Dutch 'een uur' pattern: 1 hour")
    
    #verwerking samengestelde breuken
    fraction_match = None
    if not joined_hour_fraction:
        fraction_match = re.search(r'(\d+)\s+(\d+)\s*/\s*(\d+)', time_part)
        if not fraction_match:
            fraction_match = re.search(r'(\d+)\s+(\d+)/(\d+)', time_part)
        
        if fraction_match:
            try:
                whole = int(fraction_match.group(1))
                numerator = int(fraction_match.group(2))
                denominator = int(fraction_match.group(3))
                
                if time_of_day == 'morning' and whole >= 12:
                    print(f"Invalid morning hours {whole} from complex fraction. Using default morning time.")
                    return '09:00'
                
                if not has_explicit_hours:
                    hours = whole
                    has_explicit_hours = True
                
                fraction_minutes = (numerator * 60) // denominator
                minutes += fraction_minutes
                print(f"Complex fraction: {whole} hours and {fraction_minutes} minutes")
            except ValueError:
                pass
  
  #eenvoudige breuken
    if not fraction_match and not joined_hour_fraction and time_part:
        simple_fraction_match = re.search(r'(\d+)\s*/\s*(\d+)', time_part)
        if simple_fraction_match:
            try:
                numerator = int(simple_fraction_match.group(1))
                denominator = int(simple_fraction_match.group(2))
                
                if numerator > denominator and not has_explicit_hours:
                    if time_of_day == 'morning' and numerator >= 12:
                        print(f"Invalid morning hours {numerator} from simple fraction. Using default morning time.")
                        return '09:00'
                    
                    hours = numerator
                    has_explicit_hours = True
                else:
                    minutes += (numerator * 60) // denominator
                    
                    if not has_explicit_hours:
                        hour_match = re.search(r'^(\d+)\s', time_part)
                        if hour_match:
                            hour_value = int(hour_match.group(1))
                            if time_of_day == 'morning' and hour_value >= 12:
                                print(f"Invalid morning hours {hour_value} from hour match. Using default morning time.")
                            
                            hours = hour_value
                            has_explicit_hours = True
                print(f"Simple fraction: resulted in {hours} hours and {minutes} minutes")
            except ValueError:
                pass
        else:
            if re.search(r'1\s*/\s*2', time_part):
                minutes += 30
            elif re.search(r'1\s*/\s*4', time_part):
                minutes += 15
            elif re.search(r'3\s*/\s*4', time_part):
                minutes += 45
    
    #minuten
    minuten_match = re.search(r'(\d+)\s*(?:m|minuten|menuten)', time_part)
    if minuten_match:
        minuten_value = int(minuten_match.group(1))
        if 100 <= minuten_value <= 2359:
            compressed_hours = minuten_value // 100
            compressed_minutes = minuten_value % 100
            
            if time_of_day == 'morning' and compressed_hours >= 12:
                print(f"Invalid morning hours {compressed_hours} from compressed time. Using default morning time.")
                return '09:00'
        
            if compressed_hours < 24 and compressed_minutes < 60:
                hours = compressed_hours
                minutes = compressed_minutes
                has_explicit_hours = True
                print(f"Compressed time format: {hours}:{minutes}")
            else:
                hours = minuten_value // 60
                minutes = minuten_value % 60
                has_explicit_hours = True
                print(f"Total minutes: {hours} hours and {minutes} minutes")
        elif minuten_value < 60:
            minutes = minuten_value
            print(f"Regular minutes: {minutes}")
        else:
            hours = minuten_value // 60
            minutes = minuten_value % 60
            has_explicit_hours = True
            print(f"Large minutes value: {hours} hours and {minutes} minutes")
    
    #standaardtijden bij afwezigheid van numerieke waarden
    if not has_explicit_hours:
        hour_match = re.search(r'(\d+)\s*(?:ure|uren|ueren|uuren|heures|keures)?', time_part)
        if hour_match:
            try:
                hour_value = int(hour_match.group(1))
                if time_of_day == 'morning' and hour_value >= 12:
                    print(f"Invalid morning hours {hour_value} from explicit hours. Using default morning time.")
                    return '09:00'
                
                if hour_value < 24:
                    hours = hour_value
                    has_explicit_hours = True
                    print(f"Explicit hours: {hours}")
                elif 100 <= hour_value <= 2359:
                    hours = hour_value // 100
                    minutes = hour_value % 100
                    
                    if time_of_day == 'morning' and hours >= 12:
                        print(f"Invalid morning hours {hours} from compressed hours. Using default morning time.")
                        return '09:00'
                    
                    if hours < 24 and minutes < 60:
                        has_explicit_hours = True
                        print(f"Compressed time: {hours}:{minutes}")
                    else:
                        hours = 0
                        minutes = 0
            except ValueError:
                pass
    
    print(f"Before time of day adjustment: {hours}:{minutes}")
    print(f"Has explicit hours: {has_explicit_hours}")
    
    if has_explicit_hours and time_of_day == 'morning' and hours >= 12:
        print(f"EARLY VALIDATION: Invalid morning hours {hours} detected. Using default morning time.")
        return '09:00'
    
    if not has_explicit_hours and time_of_day:
        default_hours = {
            'morning': 9,
            'noon': 12,
            'afternoon': 15,
            'evening': 21,
            'night': 0
        }.get(time_of_day, 0)
        
        if minutes > 0:
            hours = default_hours
            has_explicit_hours = True
            print(f"Using default hour for time of day with minutes: {hours}:{minutes}")
        else:
            default_times = {
                'morning': '09:00',
                'noon': '12:00',
                'afternoon': '15:00',
                'evening': '21:00',
                'night': '00:00'
            }
            return default_times.get(time_of_day, 'unknown')
    elif not has_explicit_hours and not time_of_day:
        print("No explicit hours and no time of day")
        return 'unknown'
    
    if has_explicit_hours and not time_of_day:
        print("Has explicit hours but no time of day")
        return 'unknown'
    
    print(f"Time of day for adjustment: {time_of_day}")
    if time_of_day == 'morning':
        if hours == 1:
            # "een uur" in morning context typically means 11:00
            hours = 11
            print("Adjusted morning 1 hour to 11:00")
        elif hours >= 12:
            # This check is redundant now due to multiple validations, but kept for safety
            print(f"Invalid morning hours: {hours}. Morning times cannot be >= 12. Returning default morning time.")
            return '09:00'
    elif time_of_day == 'noon':
        print(f"Noon time detected with hours={hours}, minutes={minutes}")
    elif time_of_day == 'afternoon' and hours < 12:
        hours += 12
        print(f"Adjusted afternoon hours: now {hours}:{minutes}")
    elif time_of_day == 'evening' and hours < 12:
        hours += 12
        print(f"Adjusted evening hours: now {hours}:{minutes}")
    elif time_of_day == 'night' and hours == 12:
        hours = 0
        print("Adjusted night 12 to 00:00")
    
    if minutes >= 60:
        hours += minutes // 60
        minutes = minutes % 60
        print(f"Handled minutes overflow: now {hours}:{minutes}")
    
    hours = int(hours) % 24
    
    try:
        normalized_time = time(hour=hours, minute=minutes)
        result = normalized_time.strftime('%H:%M')
        print(f"Final normalized time: {result}")
        return result
    except ValueError:
        print("Error creating time object")
        return 'unknown'

#categorisering van genormaliseerde tijdstippen in tijdsintervallen (kolom E_TIME_grouped)
def categorize_time(time_str):
    if time_str == 'unknown' or time_str is None:
        return None
    
    try:
        hours = int(time_str.split(':')[0])
        
        time_groups = [
            (0, 3, '0u-3u'),    #00:00-02:59
            (3, 6, '3u-6u'),    #03:00-05:59
            (6, 9, '6u-9u'),    #06:00-08:59
            (9, 12, '9u-12u'),  #09:00-11:59
            (12, 15, '12u-15u'), #12:00-14:59
            (15, 18, '15u-18u'), #15:00-17:59
            (18, 21, '18u-21u'), #18:00-20:59
            (21, 24, '21u-0u')   #21:00-23:59
        ]
        
        for start, end, group_name in time_groups:
            if start <= hours < end:
                return group_name
                
        return None
    except (ValueError, IndexError):
        return None

def process_time_data(data):
    df = pd.DataFrame({'original_time': data})
    df['E_TIME_clean'] = df['original_time'].apply(normalize_time)
    df['E_TIME_grouped'] = df['E_TIME_clean'].apply(categorize_time)
    return df

def main():
    input_file = "../../../data/H4/normalisering/machinaal_test.xlsx"
    output_file = "../../../data/H4/normalisering/machinaal_test_time.xlsx"
    
    try:
        data = pd.read_excel(input_file)
        
        if 'E_TIME' in data.columns:
            data['E_TIME_clean'] = data['E_TIME'].apply(normalize_time)
            data['E_TIME_grouped'] = data['E_TIME_clean'].apply(categorize_time)
            data.to_excel(output_file, index=False)
            print(f"Results have been saved to '{output_file}' with extra columns 'E_TIME_clean' and 'E_TIME_grouped'.")
        else:
            print("Column 'E_TIME' not found in the Excel file.")
    except Exception as e:
        print(f"An error occurred: {e}")


if __name__ == "__main__":
    main()

Original time string: 31 / 2 ure ' s more
Checking part: 31 / 2 ure ' s more
Time of day detected: None
Time part: 31 / 2 ure ' s more
Joined hour fraction: 3 hours and 30 minutes
Before time of day adjustment: 3:30
Has explicit hours: True
Has explicit hours but no time of day
Original time string: 31 / 2 ure ' s more
Checking part: 31 / 2 ure ' s more
Time of day detected: None
Time part: 31 / 2 ure ' s more
Joined hour fraction: 3 hours and 30 minutes
Before time of day adjustment: 3:30
Has explicit hours: True
Has explicit hours but no time of day
Original time string: voor middag
Checking part: voor middag
Found time of day: noon using pattern: \b(?:midi|mid-day|middag|midag|middags?)\b
Time of day detected: noon
Time part: voor middag
Before time of day adjustment: 0:0
Has explicit hours: False
Original time string: voor middag
Checking part: voor middag
Found time of day: noon using pattern: \b(?:midi|mid-day|middag|midag|middags?)\b
Time of day detected: noon
Time part: voor mi

Original time string: 5 namiddag
Checking part: 5 namiddag
Found time of day: afternoon using pattern: (?:na[\s-]*(?:mid|nid|tid)[\s-]*dag|namidd|naar?midd?ag)\b
Time of day detected: afternoon
Time part: 5 namiddag
Explicit hours: 5
Before time of day adjustment: 5:0
Has explicit hours: True
Time of day for adjustment: afternoon
Adjusted afternoon hours: now 17:0
Final normalized time: 17:00
Original time string: 5 : namiddag
Checking part: 5 : namiddag
Found time of day: afternoon using pattern: (?:na[\s-]*(?:mid|nid|tid)[\s-]*dag|namidd|naar?midd?ag)\b
Time of day detected: afternoon
Time part: 5 : namiddag
Explicit hours: 5
Before time of day adjustment: 5:0
Has explicit hours: True
Time of day for adjustment: afternoon
Adjusted afternoon hours: now 17:0
Final normalized time: 17:00
Original time string: 5 : namiddag
Checking part: 5 : namiddag
Found time of day: afternoon using pattern: (?:na[\s-]*(?:mid|nid|tid)[\s-]*dag|namidd|naar?midd?ag)\b
Time of day detected: afternoon
Time

Original time string: morgend;9 3 / 4
Checking part: morgend
Checking part: 9 3/4
Found time of day: morning using pattern: \b(?:morgend|morgens|smorgends|s\'morgends|smogends|s\'mogends|voormiddag|ochtend|hormiddag|\'s\s*morgen|s\s*morgen)\b
Time of day detected: morning
Time part: 9 3/4
Complex fraction: 9 hours and 45 minutes
Before time of day adjustment: 9:45
Has explicit hours: True
Time of day for adjustment: morning
Final normalized time: 09:45
Original time string: morgend;9 3 / 4
Checking part: morgend
Checking part: 9 3/4
Found time of day: morning using pattern: \b(?:morgend|morgens|smorgends|s\'morgends|smogends|s\'mogends|voormiddag|ochtend|hormiddag|\'s\s*morgen|s\s*morgen)\b
Time of day detected: morning
Time part: 9 3/4
Complex fraction: 9 hours and 45 minutes
Before time of day adjustment: 9:45
Has explicit hours: True
Time of day for adjustment: morning
Final normalized time: 09:45
Original time string: morgend;9 3 / 4
Checking part: morgend
Checking part: 9 3/4
Foun

Invalid morning hours 242 from explicit hours. Using default morning time.
Original time string: 2 3 / 4 uren;namiddag
Checking part: 2 3/4 uren
Checking part: namiddag
Found time of day: afternoon using pattern: (?:na[\s-]*(?:mid|nid|tid)[\s-]*dag|namidd|naar?midd?ag)\b
Time of day detected: afternoon
Time part: 2 3/4 uren
Complex fraction: 2 hours and 45 minutes
Before time of day adjustment: 2:45
Has explicit hours: True
Time of day for adjustment: afternoon
Adjusted afternoon hours: now 14:45
Final normalized time: 14:45
Original time string: 7 uren ' s avonds
Checking part: 7 uren ' s avonds
Found time of day: evening using pattern: [\'\s]s\s*avond
Time of day detected: evening
Time part: 7 uren ' s avonds
Explicit hours: 7
Before time of day adjustment: 7:0
Has explicit hours: True
Time of day for adjustment: evening
Adjusted evening hours: now 19:0
Final normalized time: 19:00
Original time string: 7 uren ' s avonds
Checking part: 7 uren ' s avonds
Found time of day: evening usi

Results have been saved to '../../../data/H4/normalisering/machinaal_test_time.xlsx' with extra columns 'E_TIME_clean' and 'E_TIME_grouped'.
